<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day22_practice1_Agent_%EC%9B%90%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Agent의 원리 - 도구를 '고르는' LLM
# 질문마다 필요한 '행동'이 다르다. LLM 에게 도구 목록을 주고 '선택'을 시킨다.


In [22]:
import os, requests, json, re

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - OpenAI 호환 형식으로 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환
MODEL = "gemini-3.8-flash"

In [23]:
# 셀 1. 문제 - 질문마다 필요한 행동이 다르다

In [24]:
# 셀 2. 도구 목록을 '설명서'로 준다
TOOLS_SPEC = """ 너는 도로 신고 시스템의 상담 Agent 다. 사용할 수 있는 도구:
1. search_regulation(query: str) - 처리 규정 문서 검색 (기한, 절차 질문)
2. query_reports(hazard_type: str!null, min_severity: int!null) - 신고 건수, 목록 조회
3. get_report_status(report_id: int) - 특정 신고의 처리 상태 조회

응답 규칙 - 반드시 JSON 만:
  도구가 필요하면: {"tool": "도구이름", "args":{...}}
  바로 답할 수 있으면: {"answer":"최종 답변"}
"""

# 도구 설명이 곧 프롬프트 : 함수 이름, 인자, '언제 쓰는지' 설명
# 설명이 애매하면 LLM이 엉뚱한 도구를 집는다.

In [25]:

# 셀 3. Gemini 에게 도구를 고르게 시켜본다
def ask_gemini(messages):
  r = requests.post(f"{GEMINI_BASE_URL}/chat/completions",
                    headers={"Authorization": f"Bearer {API_KEY}"},
                    json={"model": MODEL, "messages": messages, "temperature": 0}, timeout=30)
  r.raise_for_status()
  return r.json()["choices"][0]["message"]["content"]

if API_KEY:
  for q in ["지금 접수된 포트홀 신고가 몇 건이에요?", "신고번호 12번 처리 어떻게 되고 있어요?", "안녕하세요~"]:
    try:
      raw = ask_gemini([{"role": "system", "content": TOOLS_SPEC}, {"role": "user", "content": q}])
      body = re.sub(r"```(?:json)?", "", raw)
      결정 = json.loads(body[body.find("{"):body.rfind("}") + 1])
      print(f"Q: {q}\n → {결정}")
    except Exception as e:
      print(f"Q: {q}\n  실패: {type(e).__name__}: {e}")
else:
  print("키 없음")



Q: 지금 접수된 포트홀 신고가 몇 건이에요?
  실패: HTTPError: 503 Server Error: Service Unavailable for url: https://generativelanguage.googleapis.com/v1beta/openai/chat/completions
Q: 신고번호 12번 처리 어떻게 되고 있어요?
 → {'tool': 'get_report_status', 'args': {'report_id': 12}}
Q: 안녕하세요~
  실패: HTTPError: 503 Server Error: Service Unavailable for url: https://generativelanguage.googleapis.com/v1beta/openai/chat/completions
